# ARGUS · live training demo

> **A polygraph for self-improving AI.**
> The environment that catches an AI lying to itself about whether it's learning.

This Colab runs the entire ARGUS loop end-to-end on a free T4 in **≈10 min**. Same code as the deployed
Hugging Face Space — same chain-consensus reward, same metacognition stack, same lie taxonomy.

**Why this notebook matters**

The headline 15-episode run on a real GPU produced three different stories — `+7 pp`, `−5 pp`, `+2.5 pp`
— with a 3-seed mean inside the Wilson 95% noise band. **The metric is sampling-bound; the
*architecture* is what reproduces.** This notebook shows that architecture working live.

**What you'll see**
1. The env boots and the chain-consensus reward becomes the actual training signal.
2. As self-play runs, **lie-type events print live** — Type B (novelty collapse), Type D (forgetting),
   per-cluster firings, ERCV refusals.
3. Pre / post pass@1 on GSM8K with a delta you can compare to the headline runs.
4. A final dashboard plots reward, skill, and how your run sits vs the three published seeds.

| resource | link |
| --- | --- |
| Live interactive demo | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space](https://vaibhav-pandeyy-argus-self-learning-env.hf.space) |
| HF Space repo | [https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV](https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV) |
| GitHub | [https://github.com/VP-Vaibhav-Pandey/argus-env](https://github.com/VP-Vaibhav-Pandey/argus-env) |
| Architecture | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space/architecture](https://vaibhav-pandeyy-argus-self-learning-env.hf.space/architecture) |
| The exact training scripts (3 runs) | [https://github.com/VP-Vaibhav-Pandey/argus-env/tree/main/training](https://github.com/VP-Vaibhav-Pandey/argus-env/tree/main/training) |

**Before running**: `Runtime` → `Change runtime type` → `T4 GPU`. Then `Runtime` → `Run all`.

---


## 1 · Pre-flight check

One cell, three checks: GPU available, internet works, dataset reachable.
If anything fails, fix it before continuing — saves you a 10-min wasted run.

In [ ]:
import sys, subprocess, urllib.request

print('▸ python ', sys.version.split()[0])

# GPU
try:
    import torch
    if torch.cuda.is_available():
        print(f'✓ GPU    {torch.cuda.get_device_name(0)} · '
              f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('✗ GPU    NO GPU. Runtime → Change runtime type → T4 GPU')
        raise SystemExit(1)
except ImportError:
    print('▸ torch not yet installed (will install in next cell)')

# Network
try:
    urllib.request.urlopen('https://huggingface.co', timeout=5).close()
    print('✓ net    huggingface.co reachable')
except Exception as e:
    print(f'✗ net    HF unreachable: {e}')

print()
print('all green → continue to cell 2')


## 2 · Install dependencies

≈90 s on a cold Colab session.

In [ ]:
%%capture
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets huggingface_hub
!pip install -q sentencepiece numpy scipy scikit-learn matplotlib


## 3 · Clone the env

The deployed HF Space is itself a git repository. We clone it directly — same code that serves the live demo at <https://vaibhav-pandeyy-argus-self-learning-env.hf.space>.

In [ ]:
import os
REPO_DIR = 'argus'
HF_SPACE_GIT = 'https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV'

if not os.path.exists(REPO_DIR):
    !git clone -q {HF_SPACE_GIT} {REPO_DIR}
%cd {REPO_DIR}
!ls -1 src/env/ src/control/ src/eval/ 2>/dev/null | head -20


## 4 · Config · the only knob you need

Two switches control the entire run. Flip `METACOG_ON` to `False` to reproduce the no-metacognition
ablation (the +2.5 pp control run). Bump `EPISODES` to see more architecture activity.

In [ ]:
# === ARGUS run config — change these to compare regimes ===
METACOG_ON   = True    # True = full v3.4 stack · False = ablation (no capability map / no per-cluster detectors)
EPISODES     = 2       # 2 ep × 24 steps × 4 chains ≈ 9 min on T4. Set to 1 for ~5 min.
STEPS_PER_EP = 24
K_CHAINS     = 4
EVAL_N       = 10      # held-out GSM8K eval size
SEED         = 0

print(f'mode      : {"FULL v3.4" if METACOG_ON else "ABLATION (no metacog)"}')
print(f'episodes  : {EPISODES} × {STEPS_PER_EP} steps · {K_CHAINS} chains/problem')
print(f'eval_n    : {EVAL_N}')
print(f'seed      : {SEED}')


## 5 · Load model + LoRA

Qwen2.5-1.5B-Instruct in 4-bit. Same architecture family as the headline run, just smaller and
universally available so the Colab cold-starts in seconds.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
)
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


## 6 · Boot ARGUS in-process

Same env code as the deployed Space, just imported here. No HTTP round-trips — fastest path for a Colab
demo. (See the optional cell at the end for hitting the deployed Space over HTTP instead.)

In [ ]:
import sys
sys.path.insert(0, '.')
from src.env.self_improvement_env import make_env, chain_consensus_score

env = make_env(
    seed=SEED,
    solver_k=K_CHAINS,
    frontier_low=0.4,
    frontier_high=0.8,
    retest_every=4,
    memory_size=64,
)
obs, info = env.reset(seed=SEED)
print(f'env initialised · mode={obs["mode"]}  skill={obs["skill_level"]:.3f}')


## 7 · Pre-eval baseline · pass@1 on GSM8K

In [ ]:
import re
from datasets import load_dataset

gsm = load_dataset('gsm8k', 'main', split='test').shuffle(seed=42).select(range(EVAL_N))

def extract_answer(text):
    m = re.search(r'####\s*(-?\d+(?:[\.,]\d+)?)', text or '')
    if m: return m.group(1).replace(',', '')
    nums = re.findall(r'-?\d+(?:\.\d+)?', text or '')
    return nums[-1] if nums else ''

def gold(answer_field):
    m = re.search(r'####\s*(-?\d+(?:[\.,]\d+)?)', answer_field)
    return m.group(1).replace(',', '') if m else ''

def generate(prompt, max_new=192, n=1, temperature=0.8):
    msgs = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors='pt').to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new,
        num_return_sequences=n,
        pad_token_id=tokenizer.pad_token_id,
    )
    if temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        gen_kwargs.update(do_sample=False)  # greedy
    out = model.generate(**inp, **gen_kwargs)
    return [tokenizer.decode(o[inp['input_ids'].shape[1]:], skip_special_tokens=True) for o in out]

def eval_passat1():
    correct = 0
    for ex in gsm:
        q = ex['question']
        prompt = q + "\n\nSolve step by step. End your answer with '#### N' where N is the final number."
        chain = generate(prompt, n=1, temperature=0.0, max_new=192)[0]
        if extract_answer(chain) == gold(ex['answer']):
            correct += 1
    return correct / EVAL_N

pre = eval_passat1()
print(f'pre · pass@1 (n={EVAL_N}) = {pre:.3f}')


## 8 · Self-play · the architecture firing live

The env alternates `propose` and `solve` modes. Below, we capture **every defensive event** the
architecture produces — chain-consensus zones, lie-type detections, ERCV decisions — and print them
with color codes as they happen. This is the metacognition layer working.

Watch for:
- 🟢 `frontier` zone — the productive learning band (the env trains on these)
- 🟠 `easy` / 🔴 `hard` — outside the frontier, lower training weight
- ⚠ `LIE` — a typed failure-mode detector firing
- 🛑 `ERCV` — a soft-rollback (training step refused)


In [ ]:
from src.eval.lie_taxonomy import classify_episode
import time

# ANSI color helpers
def _c(text, color):
    return f'\033[{color}m{text}\033[0m'
def green(t): return _c(t, '32')
def amber(t): return _c(t, '33')
def red(t):   return _c(t, '31')
def gray(t):  return _c(t, '90')
def bold(t):  return _c(t, '1')

# Storage for the final dashboard
training_triples = []
ep_history      = []
event_log       = []   # list of (ep, step, kind, detail) tuples
zone_counts     = {'easy': 0, 'frontier': 0, 'hard': 0}

def log_event(ep, step, kind, detail):
    event_log.append((ep, step, kind, detail))

print(bold(f'▶ self-play · {EPISODES} episodes × {STEPS_PER_EP} steps'))
print(gray('─' * 72))

for ep in range(EPISODES):
    obs, _ = env.reset(seed=SEED + ep)
    step_rewards, frontier_steps = [], 0
    skills, rewards = [], []
    t0 = time.time()
    for step in range(STEPS_PER_EP):
        if obs['mode'] == 'propose':
            seed_text = obs.get('context_seed') or ''
            prompt = (
                f"Generate a NEW grade-school math word problem (~{obs.get('frontier_hint', 3)} "
                f"reasoning steps).\n\nReference: {seed_text[:200]}\n\nOutput only the problem."
                if seed_text else
                f"Generate a grade-school math word problem (~{obs.get('frontier_hint', 3)} steps). "
                f"Output only the problem."
            )
            problem = generate(prompt, max_new=96, n=1, temperature=1.0)[0].strip()
            obs, r, term, trunc, info = env.step(problem)
        else:  # solve
            problem_text = obs['problem']  # capture BEFORE step (mode switches)
            chains = generate(problem_text, max_new=96, n=K_CHAINS, temperature=0.8)
            obs, r, term, trunc, info = env.step(chains)
            step_rewards.append(float(r))
            skills.append(float(obs.get('skill_level', 0.0)))
            rewards.append(float(r))
            zone = info.get('zone')
            if zone in zone_counts: zone_counts[zone] += 1

            cs = chain_consensus_score(chains)
            cs_val = cs.get('combined', 0.0)
            zone_str = {'easy': green('easy    '), 'frontier': amber('frontier'), 'hard': red('hard    ')}.get(zone, gray('?       '))
            print(f'  ep{ep+1} step{step:>2}  {zone_str}  cs={cs_val:.2f}  '
                  f'skill={obs.get("skill_level", 0):.3f}  '
                  f'r={r:+.2f}')

            if zone == 'frontier':
                frontier_steps += 1
                best_idx = cs.get('best_chain_idx', 0) or 0
                training_triples.append((problem_text, chains[best_idx], float(r)))

            # Print any defensive events surfaced by the env
            if info.get('lie_firing'):
                lie_t = info.get('lie_type', '?')
                lie_score = info.get('lie_score', 0)
                msg = f'    {amber("⚠ LIE")} type={lie_t} score={lie_score:.2f}'
                print(msg)
                log_event(ep+1, step, 'lie', f'type={lie_t} score={lie_score:.2f}')
            if info.get('ercv_refused'):
                z = info.get('ercv_z', 0)
                msg = f'    {red("🛑 ERCV REFUSED")} z={z:+.2f}'
                print(msg)
                log_event(ep+1, step, 'ercv', f'z={z:+.2f}')

        if term or trunc:
            break

    summary = {
        'ep': ep+1,
        'mean_reward': sum(step_rewards) / max(1, len(step_rewards)),
        'final_skill': skills[-1] if skills else 0.0,
        'frontier_steps': frontier_steps,
        'pairs': len(training_triples),
        'rewards': rewards,
        'skills': skills,
    }
    ep_history.append(summary)
    dt = time.time() - t0
    print(gray('─' * 72))
    print(bold(f'  ep{ep+1} done · mean_reward={summary["mean_reward"]:.3f} · '
               f'frontier={frontier_steps} · pairs={len(training_triples)} · {dt:.0f}s'))
    print(gray('─' * 72))

print()
print(bold(f'collected: {len(training_triples)} weighted (problem, chain) triples'))
print(f'zones · easy {zone_counts["easy"]} · frontier {zone_counts["frontier"]} · hard {zone_counts["hard"]}')
print(f'events · {sum(1 for e in event_log if e[2]=="lie")} lies caught · '
      f'{sum(1 for e in event_log if e[2]=="ercv")} ERCV refusals')


## 9 · Train with TRL `SFTTrainer` · consensus-weighted SFT

Standard TRL SFT, with one twist: the per-sample loss is multiplied by its chain-consensus weight.
This is **advantage-weighted regression** — high-reward chains contribute more to the gradient than
low-reward ones.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import torch.nn.functional as F

def format_pair(problem, chain):
    msgs = [
        {'role': 'user', 'content': problem},
        {'role': 'assistant', 'content': chain},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

rows = [{'text': format_pair(p, c), 'weight': max(0.1, w)} for p, c, w in training_triples]
if not rows:
    raise RuntimeError('No frontier-zone pairs collected. Increase EPISODES or check env.')
ds = Dataset.from_list(rows)
print(f'training pairs: {len(ds)} · weight range: '
      f'{min(r["weight"] for r in rows):.2f} .. {max(r["weight"] for r in rows):.2f}')

def weighted_collator(batch):
    weights = torch.tensor([float(b['weight']) for b in batch], dtype=torch.float32)
    enc = tokenizer(
        [b['text'] for b in batch],
        padding=True, truncation=True, max_length=512, return_tensors='pt',
    )
    enc['labels'] = enc['input_ids'].clone()
    enc['labels'][enc['attention_mask'] == 0] = -100
    enc['weights'] = weights
    return enc

class WeightedSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        weights = inputs.pop('weights')
        labels = inputs['labels']
        out = model(**inputs)
        logits = out.logits[:, :-1].contiguous()
        targets = labels[:, 1:].contiguous()
        nll = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            reduction='none', ignore_index=-100,
        ).view(targets.shape[0], -1)
        mask = (targets != -100).float()
        per_sample = (nll * mask).sum(1) / mask.sum(1).clamp(min=1)
        w = weights.to(per_sample.device, dtype=per_sample.dtype)
        loss = (per_sample * w).sum() / w.sum().clamp(min=1e-6)
        return (loss, out) if return_outputs else loss

cfg = SFTConfig(
    output_dir='argus_run',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='no',
    bf16=True,
    report_to='none',
    remove_unused_columns=False,
)
trainer = WeightedSFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds,
    data_collator=weighted_collator,
)
trainer.train()


## 10 · Post-eval · pass@1 after ARGUS-gated training

In [ ]:
post = eval_passat1()
delta = post - pre

print()
print('  ' + '─'*58)
print(f'  ARGUS Colab demo · pass@1 result on n={EVAL_N}')
print('  ' + '─'*58)
print(f'    pre   :  {pre:.3f}')
print(f'    post  :  {post:.3f}')
print(f'    delta :  {delta:+.3f}')
print('  ' + '─'*58)


## 11 · Final dashboard · your run vs the headline runs

Two outputs in one cell:
- A reward + skill curve from your self-play
- A comparison table: your delta vs **seed 0** (+7 pp), **seed 1** (−5 pp), **no-metacog** (+2.5 pp).

The headline 3-run mean is +1.5 pp inside the Wilson noise band — *the metric is sampling-bound;
the architecture is what reproduces.* Your single-cell Colab run is a sample from that same noise.

In [ ]:
import matplotlib.pyplot as plt

# ===== Plot reward + skill across all collected steps =====
all_rewards, all_skills = [], []
for s in ep_history:
    all_rewards += s['rewards']
    all_skills  += s['skills']

fig, ax1 = plt.subplots(figsize=(8, 3.5), dpi=100)
fig.patch.set_facecolor('#000000'); ax1.set_facecolor('#0E0E10')
ax2 = ax1.twinx(); ax2.set_facecolor('#0E0E10')

steps = list(range(1, len(all_rewards)+1))
ax1.plot(steps, all_rewards, color='#0A84FF', linewidth=2, marker='o', markersize=4, label='reward (chain consensus)')
ax2.plot(steps, all_skills,  color='#30D158', linewidth=2, linestyle='--', marker='s', markersize=3, label='skill estimate')

for ax in (ax1, ax2):
    for spine in ax.spines.values(): spine.set_color('#3F3F46')
    ax.tick_params(colors='#8E8E93')
ax1.set_xlabel('step (across all episodes)', color='#B4B4BD', fontsize=10)
ax1.set_ylabel('reward · chain consensus', color='#0A84FF', fontsize=10)
ax2.set_ylabel('skill (in-loop)', color='#30D158', fontsize=10)
ax1.set_title('Your run · reward + skill across self-play', color='#F5F5F7', fontsize=12, loc='left')
ax1.grid(True, color='#27272A', linewidth=0.5)
plt.tight_layout()
plt.show()

# ===== Comparison table to the three published runs =====
print()
print('  ' + '─'*70)
print('  Your Colab run vs the three published 15-episode runs')
print('  ' + '─'*70)
print(f'  {"run":24}  {"Δ pass@1":>12}  {"defensive events":>18}  {"discovery":>12}')
print(f'  {"─"*24}  {"─"*12}  {"─"*18}  {"─"*12}')
print(f'  {"this notebook":24}  {delta:>+12.3f}  {"".rjust(18)}  {"".rjust(12)}')
print(f'  {"seed 0 · full v3.4":24}  {0.070:>+12.3f}  {19:>18}  {"Type G":>12}')
print(f'  {"seed 1 · full v3.4":24}  {-0.050:>+12.3f}  {16:>18}  {"Type H":>12}')
print(f'  {"ablation · no-metacog":24}  {0.025:>+12.3f}  {4:>18}  {"—":>12}')
print(f'  {"3-seed mean":24}  {0.015:>+12.3f}  {"(within noise)":>18}  {"—":>12}')
print('  ' + '─'*70)
print()
print(f'  Wilson 95% noise band on n=100: ±13.9 pp.')
print(f'  Your n={EVAL_N} run has a much wider band — read the delta as direction, not magnitude.')
print()
print(f'  events from your run:')
print(f'    {sum(1 for e in event_log if e[2]=="lie")} lie firings · '
      f'{sum(1 for e in event_log if e[2]=="ercv")} ERCV refusals')
print(f'    zones · easy {zone_counts["easy"]} · frontier {zone_counts["frontier"]} · hard {zone_counts["hard"]}')


## 12 · Save the trained adapter

Saves your LoRA weights to `/content/argus_adapter/` for download. Optional: push to HF Hub by
filling in your username + token below.

In [ ]:
from pathlib import Path

ADAPTER_DIR = '/content/argus_adapter'
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'✓ adapter saved to {ADAPTER_DIR}')
print(f'  download: open the file browser (left sidebar) → /content/argus_adapter')

# ===== Optional · push to HF Hub =====
# Set HF_USER and HF_TOKEN, uncomment, and run.
# HF_USER  = 'your-hf-username'
# HF_TOKEN = 'hf_...'
# from huggingface_hub import login, create_repo, upload_folder
# login(token=HF_TOKEN)
# repo_id = f'{HF_USER}/argus-colab-adapter'
# create_repo(repo_id, exist_ok=True, repo_type='model')
# upload_folder(folder_path=ADAPTER_DIR, repo_id=repo_id, repo_type='model')
# print(f'✓ pushed to https://huggingface.co/{repo_id}')


## 13 · Optional · drive the deployed Space over HTTP

The env you trained against is the same env serving the public demo. Uncomment to drive it remotely
with the OpenEnv contract — useful for proving the deployed Space is real and reachable.

In [ ]:
# import requests
# SPACE = 'https://vaibhav-pandeyy-argus-self-learning-env.hf.space'
#
# r = requests.post(f'{SPACE}/reset', json={'seed': 0}); r.raise_for_status()
# session_id = r.json()['session_id']
# print('session:', session_id)
#
# r = requests.post(f'{SPACE}/step', json={
#     'session_id': session_id,
#     'action': 'A bakery sells cupcakes for $3 and brownies for $2. '
#               'On Monday, 24 cupcakes and 18 brownies were sold. '
#               'How much money did the bakery make on Monday?'
# })
# print(r.json())


## Summary

You just ran the ARGUS self-improvement loop end-to-end:

1. **Loaded** Qwen2.5-1.5B-Instruct + LoRA in 4-bit
2. **Booted** ARGUS in-process (same code as the deployed Space)
3. **Pre-eval** GSM8K pass@1 baseline (n={EVAL_N})
4. **Self-play** with **live event capture** — every chain-consensus zone, lie firing, and ERCV decision
5. **SFT** with per-sample-weighted loss (advantage-weighted regression)
6. **Post-eval** to measure delta + plotted your reward / skill curve
7. **Compared** your run to the three headline 15-episode runs (seed 0 / seed 1 / no-meta)

**The core idea** — ARGUS scores reasoning chains using **outcome × process consensus**, refuses any
training step where the local "did I learn" signal disagrees with empirical retest (ERCV), and explains
which concept regressed via causal attribution. **Better defenses surface deeper failure modes** — F, G,
H were all discovered during training, not designed in advance.

| read more | link |
| --- | --- |
| Live interactive writeup | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space](https://vaibhav-pandeyy-argus-self-learning-env.hf.space) |
| OpenEnv API docs | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space/docs](https://vaibhav-pandeyy-argus-self-learning-env.hf.space/docs) |
| Architecture · 3-loop control + lie taxonomy | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space/architecture](https://vaibhav-pandeyy-argus-self-learning-env.hf.space/architecture) |
| HF Space (source) | [https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV](https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV) |
| GitHub | [https://github.com/VP-Vaibhav-Pandey/argus-env](https://github.com/VP-Vaibhav-Pandey/argus-env) |
| The exact training scripts (3 runs) | [https://github.com/VP-Vaibhav-Pandey/argus-env/tree/main/training](https://github.com/VP-Vaibhav-Pandey/argus-env/tree/main/training) |
